# RAGTIME Single-Query Retrieval

This notebook uses `RagtimeRetriever` to send one cached query to the private RAGTIME `POST /pipeline` endpoint, then batch-hydrate the ranked IDs with original-language text and metadata from the public Hugging Face dataset. Search and document responses share a cache outside the repository, so different worktrees reuse them. It does not call the private `/content` endpoint by default.

Decrypt the repository credentials before starting Jupyter:

```bash
sops decrypt --output .env config/secrets.sops.env
uv run jupyter lab notebooks/ragtime_single_query.ipynb
```

Change `QUERY` below, then run the cells in order.

In [ ]:
from textwrap import shorten

import pandas as pd
from IPython.display import display
from trec_ragtime import DEFAULT_PIPELINE, RagtimeRetriever

PIPELINE = DEFAULT_PIPELINE
QUERY = "health benefits and risks of Nordic walking for older adults"
TOP_K = 20

In [ ]:
with RagtimeRetriever.from_env() as retriever:
    result = retriever.retrieve(QUERY, pipeline=PIPELINE, top_k=TOP_K)
    shared_cache_path = retriever.cache_path

records = result.as_records(include_text=False)
for record, document in zip(records, result.documents, strict=True):
    record["text_preview"] = (
        shorten(" ".join(document.text.split()), width=280, placeholder="...")
        if document.text is not None
        else None
    )
results = pd.DataFrame(records)

In [ ]:
print(f"Query: {QUERY}")
print(f"Pipeline: {result.pipeline}")
print(f"Results returned: {len(results)}")
print(f"Fully hydrated: {result.fully_hydrated}")
print(f"Search cache: {'hit' if result.search_local_cache_hit else 'miss'}")
print(f"Document cache hits: {sum(document.content_local_cache_hit is True for document in result.documents)}/{len(result.documents)}")
print(f"Shared cache path: {shared_cache_path}")
display(results)